In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os


feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
# feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
if feature_path not in sys.path:
    sys.path.append(feature_path)

from FEATURES.features import *
from PROPS_EV.calculateEVS import *
from BACKTEST.backtest import *
from MODELS.pipeline import *

### Load Model

In [2]:
PTSmodel = joblib.load('Models/xgbPTSModel.pkl')
PTSfeatures = joblib.load('Models/topPTSfeatures.pkl')
REBmodel = joblib.load('Models/xgbREBModel.pkl')
REBfeatures = joblib.load('Models/topREBfeatures.pkl')

### Load Data

In [4]:
pd.set_option('display.max_columns', None)


s25_pts = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv')
s24_pts = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_24.csv')
s25_reb = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/REB_TRAIN_25.csv')
s24_reb = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/REB_TRAIN_24.csv')

dfPTS = pd.concat([s25_pts, s24_pts]).sort_values(by='GAME_DATE')
dfREB = pd.concat([s25_reb, s24_reb]).sort_values(by='GAME_DATE')


date = '2025-03-12'
dfPTS = dfPTS[dfPTS['GAME_DATE'] < date]
dfREB = dfREB[dfREB['GAME_DATE'] < date]

dfsData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/dfs_data.csv')
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'prizepicks') & (dfsData['GAME_DATE'] == date) & (dfsData['CATEGORY'] == 'player_points')]
dfsREB = dfsData[(dfsData['BOOKMAKER'] == 'prizepicks') & (dfsData['GAME_DATE'] == date) & (dfsData['CATEGORY'] == 'player_rebounds')]

C:\Users\alexg\AppData\Local\Temp\ipykernel_26116\4133591870.py:17: DtypeWarning: Columns (11,12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  dfsData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/dfs_data.csv')


In [5]:
backtestData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/singleBookies.csv')
backtestData = backtestData[(backtestData['ODDS'] <= 200) & (backtestData['ODDS'] >= -200)]
singlePTSBookies = backtestData[(backtestData['CATEGORY'] == 'points') & (backtestData['GAME_DATE'] == date)]
singleREBBookies = backtestData[(backtestData['CATEGORY'] == 'rebounds') & (backtestData['GAME_DATE'] == date)]

### Top EVs for single bets

In [ ]:
results = single_bet(
    data=dfPTS,
    bookmakers=singlePTSBookies,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=4.5,
    stake=5,
    simulations=10000, 
    std_window=10,
    min_std=2.0,
    max_std=10.0,
    stat_col='PTS'
)
results.sort_values(by='EV%', ascending=False).head(10)

Processing single bets...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
843,Naji Marshall,fanduel,points,17.5,132,over,22.809076,1,0.902,0.098,0.431,109.26,0.83,0.41,0.21,"(14.5, 30.9)"
204,Aaron Wiggins,betmgm,points,2.5,105,over,13.897324,1,1.000,0.000,0.488,104.98,1.00,0.50,0.25,"(7.8, 20.2)"
859,Julian Champagnie,betmgm,points,3.5,105,over,12.007226,1,0.967,0.033,0.488,98.21,0.94,0.47,0.23,"(3.0, 21.2)"
864,Sandro Mamukelashvili,espnbet,points,4.5,100,over,8.591534,0,0.958,0.042,0.500,91.60,0.92,0.46,0.23,"(3.9, 13.2)"
863,Sandro Mamukelashvili,betmgm,points,4.5,100,over,8.591534,0,0.955,0.045,0.500,90.90,0.91,0.45,0.23,"(3.9, 13.2)"
344,Adem Bona,espnbet,points,6.5,-110,over,11.677994,1,0.997,0.003,0.524,90.24,0.99,0.50,0.25,"(7.8, 15.6)"
889,Nickeil Alexander-Walker,betmgm,points,7.5,-105,over,13.501509,1,0.954,0.046,0.512,86.22,0.91,0.45,0.23,"(6.6, 20.3)"
310,Jamal Shead,draftkings,points,15.5,-105,under,9.222432,1,0.048,0.952,0.512,85.91,0.90,0.45,0.23,"(2.2, 16.7)"
903,Naz Reid,betmgm,points,12.5,-105,over,20.639114,1,0.943,0.057,0.512,84.01,0.88,0.44,0.22,"(10.6, 30.7)"
313,Jamal Shead,betrivers,points,14.5,-104,under,9.222432,1,0.077,0.923,0.510,81.01,0.84,0.42,0.21,"(2.3, 16.7)"


In [9]:
results = single_bet(
    data=dfREB,
    bookmakers=singleREBBookies,
    model=REBmodel,
    features=REBfeatures,
    edge_threshold=2.0,
    stake=5,
    simulations=10000, 
    std_window=10,
    min_std=1.5,
    max_std=6.5,
    stat_col='REB'
)
results.sort_values(by='EV%', ascending=False).head(10)

Processing single bets...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
246,Sam Hauser,draftkings,rebounds,2.5,190,over,3.452911,0,0.715,0.285,0.345,107.29,0.56,0.28,0.14,"(0.4, 7.6)"
346,Jared Butler,betmgm,rebounds,4.5,130,under,2.727530,0,0.208,0.792,0.435,82.21,0.63,0.32,0.16,"(0.3, 6.8)"
797,Jeremy Sochan,betmgm,rebounds,6.5,120,under,4.640765,0,0.176,0.824,0.455,81.21,0.68,0.34,0.17,"(1.1, 8.5)"
245,Sam Hauser,betmgm,rebounds,2.5,155,over,3.452911,0,0.711,0.289,0.392,81.20,0.52,0.26,0.13,"(0.5, 7.6)"
754,Chris Paul,betmgm,rebounds,2.5,165,over,3.246181,0,0.683,0.317,0.377,80.89,0.49,0.25,0.12,"(0.4, 7.1)"
446,Mason Plumlee,betrivers,rebounds,8.5,106,under,5.970885,1,0.122,0.878,0.485,80.81,0.76,0.38,0.19,"(1.7, 10.3)"
341,Chris Boucher,betmgm,rebounds,6.5,-105,under,3.750437,1,0.085,0.915,0.512,78.72,0.83,0.41,0.21,"(0.6, 7.7)"
356,Dillon Brooks,betmgm,rebounds,3.5,135,over,4.914239,0,0.758,0.242,0.426,78.15,0.58,0.29,0.14,"(1.1, 8.9)"
542,Terry Rozier,betmgm,rebounds,2.5,135,over,3.541201,0,0.730,0.270,0.426,71.43,0.53,0.26,0.13,"(0.5, 7.6)"
443,Mason Plumlee,draftkings,rebounds,8.5,-105,under,5.970885,1,0.122,0.877,0.512,71.32,0.75,0.37,0.19,"(1.7, 10.3)"


### Top EVs for 2 leg bets

In [ ]:
results = prizepickspairsEV(
    data=dfPTS,
    bookmakers=dfsData,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=4.5,
    stake=100,
    simulations=1000,
    std_window=10,
    min_std=2.0,
    max_std=10.0,
    stat_col='PTS'
)
results.sort_values(by='EV%', ascending=False).head(10).reset_index(drop=True)

Processing pairs...


KeyboardInterrupt: 

In [ ]:
results = prizepickspairsEV(
    data=dfREB,
    bookmakers=dfsData,
    model=REBmodel,
    features=REBfeatures,
    edge_threshold=2.0,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=1.5,
    max_std=6.5,
    stat_col='REB'
)
results.sort_values(by='EV%', ascending=False).head(10).reset_index(drop=True)

## 3 leg parlay

In [ ]:
threeLeg = prizepicks3LegEV(
    data=dfPTS,
    bookmakers=dfsData,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=4.5,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=2.0,
    max_std=10.0,
    stat_col='PTS'
)
threeLeg.sort_values(by='EV%', ascending=False).head(10)

Processing 3-leg parlays...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,SIDE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,SIDE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,PLAYER 3,CATEGORY 3,BOOKMAKER 3,ODDS 3,LINE 3,SIDE 3,PREDICTION 3,MODEL_SIDE 3,OVER% 3,UNDER% 3,CONFIDENCE INTERVAL 3,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
23171,Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.062,0.938,"(2.2, 16.6)",Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.6, 33.6)",UNDER/UNDER/OVER,1,0.8272,3.963,0.793
23169,Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.062,0.938,"(2.2, 16.6)",Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Paolo Banchero,player_points,prizepicks,-137,26.5,over,17.04,UNDER,0.081,0.919,"(4.5, 30.3)",UNDER/UNDER/UNDER,1,0.8157,3.894,0.779
24548,Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Paolo Banchero,player_points,prizepicks,-137,26.5,over,17.04,UNDER,0.081,0.919,"(4.5, 30.3)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.6, 33.6)",UNDER/UNDER/OVER,1,0.8109,3.866,0.773
23590,Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.062,0.938,"(2.2, 16.6)",Paolo Banchero,player_points,prizepicks,-137,26.5,over,17.04,UNDER,0.081,0.919,"(4.5, 30.3)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.6, 33.6)",UNDER/UNDER/OVER,1,0.8033,3.820,0.764
7811,Andre Drummond,player_points,prizepicks,-137,10.5,over,6.04,UNDER,0.096,0.904,"(0.8, 12.6)",Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.062,0.938,"(2.2, 16.6)",Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",UNDER/UNDER/UNDER,0,0.8023,3.814,0.763
23178,Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.062,0.938,"(2.2, 16.6)",Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Zion Williamson,player_points,prizepicks,-137,23.0,over,26.75,OVER,0.903,0.097,"(21.2, 32.4)",UNDER/UNDER/OVER,0,0.8012,3.807,0.761
7897,Andre Drummond,player_points,prizepicks,-137,10.5,over,6.04,UNDER,0.096,0.904,"(0.8, 12.6)",Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.6, 33.6)",UNDER/UNDER/OVER,0,0.7976,3.786,0.757
24573,Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.6, 33.6)",Zion Williamson,player_points,prizepicks,-137,23.0,over,26.75,OVER,0.903,0.097,"(21.2, 32.4)",UNDER/OVER/OVER,0,0.7965,3.779,0.756
23158,Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.062,0.938,"(2.2, 16.6)",Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Kentavious Caldwell-Pope,player_points,prizepicks,-137,8.5,over,4.90,UNDER,0.103,0.897,"(0.7, 10.4)",UNDER/UNDER/UNDER,0,0.7960,3.776,0.755
24383,Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Kentavious Caldwell-Pope,player_points,prizepicks,-137,8.5,over,4.90,UNDER,0.103,0.897,"(0.7, 10.4)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.6, 33.6)",UNDER/UNDER/OVER,0,0.7914,3.748,0.750


In [ ]:
threeLeg = prizepicks3LegEV(
    data=dfREB,
    bookmakers=dfsData,
    model=REBmodel,
    features=REBfeatures,
    edge_threshold=2.0,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=1.5,
    max_std=6.5,
    stat_col='REB'
)
threeLeg.sort_values(by='EV%', ascending=False).head(10)